<a href="https://colab.research.google.com/github/Krishnan-Raghavan/Packt/blob/main/StableDiffusionChapter16.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!nvidia-smi

In [ ]:
!pip install diffusers
!pip install transformers scipy ftfy accelerate ipywidgets

In [ ]:
!pip install transformers

In [ ]:
from diffusers import AutoencoderKL

In [ ]:
vae_model = AutoencoderKL.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0"
    , subfolder = "vae"
).to("cuda:0")

In [ ]:
import torch

In [ ]:
from diffusers.utils import load_image
from diffusers.image_processor import VaeImageProcessor
image = load_image("/content/Cat1.png")

image_processor = VaeImageProcessor()
prep_image = image_processor.preprocess(image)
prep_image = prep_image.to("cuda:0")

with torch.no_grad():
    image_latent = vae_model.encode(prep_image).latent_dist.sample()

image_latent.shape

In [ ]:
with torch.no_grad():
    decode_image = vae_model.decode(
        image_latent
        , return_dict = False
    )[0]

image = image_processor.postprocess(image = decode_image)[0]
image

In [ ]:
input_prompt = "a running dog"

from transformers import CLIPTokenizer,CLIPTextModel
import torch

# initialize tokenizer 1
clip_tokenizer = CLIPTokenizer.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0"
    , subfolder = "tokenizer"
    , dtype = torch.float16
)

input_tokens = clip_tokenizer(
    input_prompt
    , return_tensors="pt"
)["input_ids"]
print(input_tokens)

clip_tokenizer_2 = CLIPTokenizer.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0"
    , subfolder = "tokenizer_2"
    , dtype = torch.float16
)

input_tokens_2 = clip_tokenizer_2(
    input_prompt
    , return_tensors="pt"
)["input_ids"]

print(input_tokens_2)

In [ ]:
clip_text_encoder = CLIPTextModel.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0"
    , subfolder = "text_encoder"
    , torch_dtype =torch.float16
).to("cuda")

# encode token ids to embeddings
with torch.no_grad():
    prompt_embeds = clip_text_encoder(
        input_tokens.to("cuda")
    )[0]

print(prompt_embeds.shape)

In [ ]:
clip_text_encoder_2 = CLIPTextModel.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0"
    , subfolder = "text_encoder_2"
    , torch_dtype =torch.float16
).to("cuda")

# encode token ids to embeddings
with torch.no_grad():
    prompt_embeds_2 = clip_text_encoder_2(input_tokens.to("cuda"))[0]

print(prompt_embeds_2.shape)

In [ ]:
from transformers import CLIPTextModelWithProjection
clip_text_encoder_2 = CLIPTextModelWithProjection.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0"
    , subfolder = "text_encoder_2"
    , torch_dtype =torch.float16
).to("cuda")

# encode token ids to embeddings
with torch.no_grad():
    pool_embed = clip_text_encoder_2(input_tokens.to("cuda"))[0]

print(pool_embed.shape)

In [ ]:
import torch
from diffusers import StableDiffusionXLPipeline
sdxl_pipe = StableDiffusionXLPipeline.from_pretrained(
    "RunDiffusion/RunDiffusion-XL-Beta"
    , torch_dtype = torch.float16
)
sdxl_pipe.watermark = None

In [ ]:
prompt = "realistic photo of astronaut cat in fighter cockpit, detailed, 8k"

sdxl_pipe.to("cuda")
image = sdxl_pipe(
    prompt                  = prompt
    , width                 = 768
    , height                = 1024
    , generator             = torch.Generator("cuda").manual_seed(1)
).images[0]

sdxl_pipe.to("cpu")
torch.cuda.empty_cache()
image

In [ ]:
from diffusers.image_processor import VaeImageProcessor
img_processor = VaeImageProcessor()

# get the size of the image
(width, height) = image.size

# upscale image
image_x = img_processor.resize(
    image = image
    , width = int(width * 1.5)
    , height = int(height * 1.5)
)
image_x

In [ ]:
from diffusers import StableDiffusionXLImg2ImgPipeline
img2img_pipe = StableDiffusionXLImg2ImgPipeline(
    vae                 = sdxl_pipe.vae
    , text_encoder      = sdxl_pipe.text_encoder
    , text_encoder_2    = sdxl_pipe.text_encoder_2
    , tokenizer         = sdxl_pipe.tokenizer
    , tokenizer_2       = sdxl_pipe.tokenizer_2
    , unet              = sdxl_pipe.unet
    , scheduler         = sdxl_pipe.scheduler
    , add_watermarker   = None
)
img2img_pipe.watermark = None

In [ ]:
img2img_pipe.to("cuda")
refine_image_2x = img2img_pipe(
    image                 = image_x
    , prompt              = prompt
    , strength            = 0.3
    , num_inference_steps = 30
    , guidance_scale      = 4.0
).images[0]

img2img_pipe.to("cpu")
torch.cuda.empty_cache()
refine_image_2x

In [ ]:
from diffusers import DiffusionPipeline
import torch

pipe = DiffusionPipeline.from_pretrained(
    "RunDiffusion/RunDiffusion-XL-Beta"
    , torch_dtype       = torch.float16
    , use_safetensors   = True
    , variant           = "fp16"
    , custom_pipeline   = "lpw_stable_diffusion_xl",
)

In [ ]:
prompt = """
glamour photography, (full body:1.5) photo of young man,
white blank background,
wear sweater, with scarf,
wear jean pant,
wear nike run shoes,
wear sun glass,
wear leather shoes,
holding a umbrella in hand
""" * 2

prompt = prompt + " a (cute cat:1.5) aside"

neg_prompt = """
(worst quality:1.5),(low quality:1.5), paint, cg, spots, bad hands,
three hands, noise, blur
"""

pipe.to("cuda")
image = pipe(
    prompt                  = prompt
    , negative_prompt       = neg_prompt
    , width                 = 832
    , height                = 1216
    , generator             = torch.Generator("cuda").manual_seed(7)
).images[0]

pipe.to("cpu")
torch.cuda.empty_cache()
image